In [1]:
import os
import shutil
import pandas as pd
from sklearn.model_selection import train_test_split

# --------- CONFIG ----------
LABELS_CSV = "./labels.csv"          # path to your labels file
IMAGES_DIR = "./images"              # folder where all images currently are
OUT_DIR = "."                        # where to create train/ and test/
TEST_SIZE = 0.2
RANDOM_STATE = 42
COPY_INSTEAD_OF_MOVE = True          # True = copy files; False = move files
# ---------------------------

TYPE_NAMES = {
    0: "No_ulcer_of_the_corneal_epithelium",
    1: "Micro_punctate",
    2: "Macro_punctate",
    3: "Coalescent_macro_punctate",
    4: "Patch_ge_1mm",
}

def ensure_dir(path: str) -> None:
    os.makedirs(path, exist_ok=True)

def main():
    df = pd.read_csv(LABELS_CSV)

    # Basic checks
    if "name" not in df.columns or "type" not in df.columns:
        raise ValueError("labels.csv must contain columns named 'name' and 'type'.")

    # Optional: clean type to int (in case it loads as float/str)
    df["type"] = df["type"].astype(int)

    # Stratified split so class proportions are similar in train/test
    train_df, test_df = train_test_split(
        df,
        test_size=TEST_SIZE,
        random_state=RANDOM_STATE,
        stratify=df["type"],
    )

    for split_name, split_df in [("train", train_df), ("test", test_df)]:
        for _, row in split_df.iterrows():
            fname = str(row["name"])          # e.g., "1.jpg"
            t = int(row["type"])              # 0..4

            # Choose folder naming style (either type_0... or readable names)
            # class_folder = f"type_{t}"
            class_folder = f"type_{t}_{TYPE_NAMES.get(t, 'unknown')}"

            src_path = os.path.join(IMAGES_DIR, fname)
            dst_dir = os.path.join(OUT_DIR, split_name, class_folder)
            dst_path = os.path.join(dst_dir, fname)

            if not os.path.exists(src_path):
                print(f"WARNING: file not found, skipping: {src_path}")
                continue

            ensure_dir(dst_dir)

            if COPY_INSTEAD_OF_MOVE:
                shutil.copy2(src_path, dst_path)
            else:
                shutil.move(src_path, dst_path)

    print("Done! Created train/ and test/ directories with class subfolders.")

if __name__ == "__main__":
    main()

Done! Created train/ and test/ directories with class subfolders.
